In [ ]:
# Una delle versioni migliori in giornata 11/05
# Feature extraction con CSP, sliding window e classificazione binaria (riposo vs attivazione). 
# Classificatore SVM lineare.

import mne
from mne.decoding import CSP
from mne_bids import BIDSPath, read_raw_bids
import numpy as np
from sklearn import preprocessing
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, cross_validate, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline, make_pipeline
from pathlib import Path
from preprocessing import create_sliding_windows
from preprocessing import create_window_labels
import matplotlib.pyplot as plt
import importlib


importlib.reload(preprocessing) # Necessario per ricaricare preprocessing se ha subito modifiche
mne.set_log_level('WARNING')

root = "../data"
root = (Path(root).resolve())
runs = ["4", "8", "12"]   # Le run che vengono prese in considerazione
train_runs = ["4", "8"]
test_run = "12" 
people = 10 # Meno 1
all_accuracy = np.zeros(people-1)  # Per memorizzare l'accuratezza di ogni soggetto

# Eseguiamo la scansione di tutti i soggetti 
for i in range(1, people):
    subject = f"{i:03d}"
    X_train = []
    y_train = []

    X_test = []
    y_test = []

    # Per ogni soggetto eseguiamo la scansione sulle run di nostro interesse
    for run in runs:
        bids_path = BIDSPath(   # Specifichiamo il percorso del dataset e BIDS eseguirà correttamente la scansione 
            subject=subject,
            task="motion",
            run=run,
            datatype="eeg",
            root=root,
        )
        
        try:
            # Fase 1: lettura dei dati
            raw = read_raw_bids(bids_path, verbose=False)  
            events, event_id = mne.events_from_annotations(raw, verbose=False) 
            raw.load_data(verbose=False) # Carico i dati in memoria per poter filtrare ecc.
            raw.filter(l_freq=8, h_freq=30, verbose=False)  # Filtro passa banda 1-30 Hz
            raw.set_eeg_reference('average', projection=False, verbose=False)  # Riferimento medio
            event_map = {
                event_id['TASK2T0']: 1,
                event_id['TASK2T1']: 2,
                event_id['TASK2T2']: 3
            }
            sfreq = raw.info['sfreq']  # frequenza di campionamento

            # Fase 2: pre-processing
            window_size = 2.0  # Lunghezza finestra in secondi
            step_size = 0.5  # Lunghezza passo in secondi

            windows, window_samples, step_samples, total_samples = create_sliding_windows(
                raw,
                window_size,
                step_size
            )        
            C_channels = ["C1", "C2", "C3", "C4", "C5", "C6", "Cz"]
            CP_channels = [ch for ch in raw.ch_names if ch.startswith("Cp")]
            FC_channels = [ch for ch in raw.ch_names if ch.startswith("Fc")]
            T_channels = [ch for ch in raw.ch_names if ch.startswith("T")]
            channels_of_interest = C_channels + CP_channels + FC_channels

            picks = mne.pick_channels(raw.ch_names, channels_of_interest)
            windows = windows[:, picks, :]

            y = create_window_labels(
                events,
                event_map,
                total_samples,
                window_samples,
                step_samples,
                threshold=0.6
            )
            
            # Divido la classificazione in due step: prima distinguo tra stato di riposo e di attivazione.
            # In caso di attivazione, distinguo tra sinistra e destra. Per ora faccio solo la prima parte.
            y_rest_active = np.where(y == 1, 0, 1)  
             
            # Variabile per addestrare il modello sul sinistra/destra (da commentare se non si vuole usare)
            mask_active = y != 1
            y_active = y[mask_active]
            y_lr = np.where(y_active == 2, 0, 1)

            # Assegno a y_binary le etichette desiderate (se voglio rest/active commento la seconda riga)
            y_binary = y_lr
            print(windows.tolist())
            windows = windows[mask_active]
            print(windows.tolist())

            if run in train_runs:
                X_train.append(windows)
                y_train.append(y_binary)
            else:
                X_test.append(windows)
                y_test.append(y_binary)

            # print(f"Soggetto {subject} run {run} - Campioni: {X_csp.shape[0]}, Feature per campione: {X_csp.shape[1]}, Etichette: {y.shape[0]}")

            # plt.scatter(
            #     X_csp[:,0],
            #     X_csp[:,1],
            #     c=y_binary
            # )

            # plt.xlabel("CSP 1")
            # plt.ylabel("CSP 2")
            # plt.show()

        except Exception as e:
            print(f"Errore {subject}: {e}")

    X_train = np.concatenate(X_train, axis=0)
    y_train = np.concatenate(y_train)

    X_test = np.concatenate(X_test, axis=0)
    y_test = np.concatenate(y_test)

    pipe = Pipeline([
        ("csp", CSP(reg='ledoit_wolf')), # reg aiuta la stabilità con finestre corte
        ("scaler", StandardScaler()),    # Fondamentale per SVM
        ("svm", SVC())
    ])

    param_grid = {
        "csp__n_components": [2, 4, 6],
        "csp__log": [True],
        "svm__C": [0.1, 1, 10, 100],
        "svm__gamma": ["scale"], 
        "svm__kernel": ["rbf"]
    }

    grid = GridSearchCV(
        pipe,
        param_grid,
        cv=5,
        scoring="balanced_accuracy",
        n_jobs=-1
    )

    grid.fit(X_train, y_train)
    print("Best score:", grid.best_score_)
    print("Best params:", grid.best_params_)

    # Test reale su run 12 
    test_accuracy = grid.score(X_test, y_test)
    all_accuracy[i-1] = test_accuracy
    print(f"Accuratezza reale su RUN 12 (Test Set): {test_accuracy:.4f}")

mid_accuracy = all_accuracy.mean()
print(f"Accuratezza media: {all_accuracy.mean()}")

